In [51]:
import os
from atproto import Client
from convenient_pickle import *
from atproto import FirehoseSubscribeReposClient, firehose_models, parse_subscribe_repos_message
from atproto import CAR, models
from atproto_client.models.network.bsky.jetstream.subscribe_events import Commit
from atproto import AsyncJetstreamClient, jetstream_models, models, IdResolver
import time
import asyncio
import queue
import base64
from sortedcontainers import SortedList
import pandas as pd
from datetime import datetime

print(time.time())

1789508514.821739


In [52]:
lookup_client = Client()
bluesky_api_key = load_pickle('api_info_do_not_upload/bluesky_api_key')
bluesky_username = load_pickle('api_info_do_not_upload/bsky_username.pkl')

In [53]:
lookup_client.login(bluesky_username, bluesky_api_key)
resolver=IdResolver()
print('')

In [28]:
client = AsyncJetstreamClient(params={'kinds':['commit']})

msg = asyncio.Queue()
outdict = dict()
messagecount = 0
stop_seconds = 1000
overtime = None
initialized = False
finished = False
last_second = -1
start_time = None
blocked_df = pd.DataFrame(columns =['collection', 'did', 'operation', 'rev', 'rkey', 'seq', 'subject','time', 'year', 'month','day','hour','minute'])



deathloopcount = 0



def unwrap_bytes(obj):
    if isinstance(obj, dict):
        if set(obj.keys()) == {'$bytes'}:
            return base64.b64decode(obj['$bytes'])
        return {k: unwrap_bytes(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [unwrap_bytes(v) for v in obj]
    return obj

async def on_message_handler(event):
    event = unwrap_bytes(event)
    #print('or maybe here?')
    try:
        await msg.put(event)
    except: 
        pass

async def stop_after_n_sec():
    global overtime
    global finished
    await asyncio.sleep(stop_seconds)
    await client.stop()
    overtime = time.time()
    await msg.put(None)
    finished = True

async def start_the_client():
    global client
    await client.start(on_message_handler)


async def pop_em_over():
    global messagecount
    
    while True:
        message = await msg.get()
        if message is None:
            break
        messagecount += 1
        if message.collection not in outdict.keys():
            outdict[message.collection] = asyncio.Queue()
            await outdict[message.collection].put(message)
        else: 
            await outdict[message.collection].put(message)

#blocked_df = pd.DataFrame(columns =['collection', 'did', 'operation', 'rev', 'rkey', 'seq', 'time', 'subject'])

async def assemble_blocked_df(): 
    global blocked_df
    while True: 
        q = outdict.get('app.bsky.graph.block')
        if q is None: 
            if finished: 
                break
            await asyncio.sleep(.05)
            continue
        try:
            blocking = await asyncio.wait_for(q.get(), timeout=0.5)
        except asyncio.TimeoutError:
            if finished and q.empty():
                break
            continue
        
        if blocking.record is not None:
            date = datetime.fromisoformat(blocking.time)
            new_row = pd.DataFrame([{
                'collection': blocking.collection,
                'did': blocking.did,
                'operation': blocking.operation,
                'rev': blocking.rev,
                'rkey': blocking.rkey,
                'seq': blocking.seq,
                'subject': blocking.record.subject,
                'time': date,
                'year': date.year,
                'month': date.month,
                'day': date.day,
                'hour': date.hour,
                'minute': date.minute
            }])
            blocked_df = pd.concat([blocked_df, new_row], ignore_index=True)

async def track_the_blockedest(): 
    global initialized
    global last_second
    while not finished: 
        if blocked_df.shape[0] > 0: 
            temp_bdf = blocked_df.copy() 
            blockedest = temp_bdf.groupby('subject')['rev'].count().sort_values(ascending=False).iloc[0:5]
            print(blockedest)
        await asyncio.sleep(10)


In [50]:
start_time = time.time()
#await asyncio.gather(stop_after_n_sec(), pop_em_over(), assemble_blocked_df(), track_the_blockedest(), start_the_client())

In [41]:
blockedest = blocked_df.copy().groupby('subject')['rev'].count().reset_index().sort_values(by='rev', ascending=False).iloc[0:5]

In [42]:
blockedest

,subject,rev
597,did:plc:5rbdy5alonu5pc2w6a662pbl,48
1504,did:plc:dmtxvda5s3ljsxzxmu5jvlsx,37
4979,did:plc:zczn76vkb7m6jtml4jrp3t32,36
1260,did:plc:c5z2qrkbvmgdq6dejgqntjxn,21
2068,did:plc:h4t7k3ctqvnlj5natv2avwpb,20


In [54]:
did_doc = resolver.did.resolve('did:plc:h4t7k3ctqvnlj5natv2avwpb')

In [55]:
did_doc

DidDocument(id='did:plc:h4t7k3ctqvnlj5natv2avwpb', also_known_as=['at://jsjjdjjsd.bsky.social'], verification_method=[VerificationMethod(id='did:plc:h4t7k3ctqvnlj5natv2avwpb#atproto', type='Multikey', controller='did:plc:h4t7k3ctqvnlj5natv2avwpb', public_key_multibase='zQ3shq2v9iuFEqpfrZCwRGyCspmHpP5DcirErxyxqp5PQtCA6')], service=[Service(id='#atproto_pds', type='AtprotoPersonalDataServer', service_endpoint='https://rhizopogon.us-west.host.bsky.network')])